# 01 - Data Exploration

Use this notebook to quickly demonstrate:
- Data loading from DuckDB
- Basic quality checks
- Price/return behavior for one ticker

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

Path("../data").mkdir(parents=True, exist_ok=True)
conn = duckdb.connect("../data/finvizaard.duckdb")
print("Connected")

In [ ]:
summary = conn.execute("""
SELECT ticker, COUNT(*) AS rows, MIN(ts) AS start_ts, MAX(ts) AS end_ts
FROM prices
GROUP BY ticker
ORDER BY rows DESC
""").df()
summary

In [ ]:
ticker = "AAPL"
df = conn.execute("""
SELECT ts, open, high, low, close, volume
FROM prices
WHERE ticker = ?
ORDER BY ts
""", [ticker]).df()

df["ret_1"] = df["close"].pct_change()
print(df.head())
print(df.describe())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(df["ts"], df["close"])
axes[0].set_title(f"{ticker} Close Price")
axes[1].plot(df["ts"], df["ret_1"])
axes[1].set_title(f"{ticker} Daily Returns")
plt.tight_layout()

## Demo Notes
- Mention rows ingested and date coverage.
- Show that returns are noisier than price level.
- This motivates regime detection and predictive models.